# HotpotQA Dataset Exploration

This notebook mirrors the GSM8K exploration workflow but focuses on the HotpotQA *distractor* split.

## How to use this notebook

1. **Setup & load HotpotQA**  
   - Import libraries, load `hotpot_qa` *distractor* split.
2. **Explore raw dataset**  
   - Look at a few examples (question, answer, context, supporting facts).
   - Inspect context structure in a small DataFrame.
   - Compute basic stats (docs/sentences per example).
3. **Inspect our preprocessing**  
   - See exactly how `preprocess_dataset` turns one example into prompt + answer.
4. **Prototype Supporting Facts (Sup) format**  
   - Build a *text* target containing `Supporting facts:` + `Answer:`.
   - Parse it back and compute simple Sup EM/F1.
5. *(Optional, later)* **Small model check (epoch 0 vs 6)**  
   - Load checkpoints and run a few validation samples with detailed debug.

You can follow these steps top‑to‑bottom to fully understand what we’re training and evaluating before we touch the main finetuning code.


In [24]:
import os
import json
from typing import List, Dict, Any

import torch
from datasets import load_dataset
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 30)

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())


2.6.0+cu124
CUDA available: True


In [25]:
# 1) Load HotpotQA distractor split
dataset = load_dataset("hotpot_qa", "distractor")
dataset


DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})

In [26]:
# 2) Peek at a few raw training examples (question, answer, context, supporting_facts)
train_ds = dataset["train"]
val_ds = dataset["validation"]
print("Train size:", len(train_ds), "Validation size:", len(val_ds))

for idx in range(3):
    ex = train_ds[idx]
    print("\n===== Example", idx, "=====")
    # In HF hotpot_qa, the field is 'id', not '_id'
    print("ID:", ex["id"])
    print("Q:", ex["question"])
    print("A:", ex["answer"])

    ctx = ex["context"]
    # HF hotpot_qa can store context either as a list of [title, sents]
    # or as a dict with 'title' and 'sentences' fields (depending on preprocessing).
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        sentences = ctx.get("sentences", [])
        ctx_pairs = list(zip(titles, sentences))
    else:
        ctx_pairs = ctx

    print("# Context docs:", len(ctx_pairs))
    # Show first 2 context docs
    for j, (title, sents) in enumerate(ctx_pairs[:2]):
        print(f"  [Doc {j}] {title} -> {len(sents)} sents")
        print("   ", " ".join(sents[:2]), "...")
    print("Supporting facts:", ex["supporting_facts"])



Train size: 90447 Validation size: 7405

===== Example 0 =====
ID: 5a7a06935542990198eaf050
Q: Which magazine was started first Arthur's Magazine or First for Women?
A: Arthur's Magazine
# Context docs: 10
  [Doc 0] Radio City (Indian radio station) -> 7 sents
    Radio City is India's first private FM radio station and was started on 3 July 2001.  It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). ...
  [Doc 1] History of Albanian football -> 4 sents
    Football in Albania existed before the Albanian Football Federation (FSHF) was created.  This was evidenced by the team's registration at the Balkan Cup tournament during 1929-1931, which started in 1929 (although Albania eventually had pressure from the teams because of competition, competition started first and was strong enough in the duels) . ...
Supporting facts: {'title': ["Arthur's Magazine", 'First for 

In [27]:
# 3) Convert a small slice to DataFrame for easier inspection
sample_df = pd.DataFrame(train_ds.select(range(10)))
# Columns in HF hotpot_qa are ['id', 'question', 'answer', ...]
sample_df[["id", "question", "answer", "context", "supporting_facts"]]


,id,question,answer,context,supporting_facts
0,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,Arthur's Magazine,"{'title': ['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', 'Women's colleges in th...","{'title': ['Arthur's Magazine', 'First for Women'], 'sent_id': [0, 0]}"
1,5a879ab05542996e4f30887e,The Oberoi family is part of a hotel company that has a head office in what city?,Delhi,"{'title': ['Ritz-Carlton Jakarta', 'Oberoi family', 'Ishqbaaaz', 'Hotel Tallcorn', 'Mohan Singh Oberoi', 'Hotel Bond...","{'title': ['Oberoi family', 'The Oberoi Group'], 'sent_id': [0, 0]}"
2,5a8d7341554299441c6b9fe5,"Musician and satirist Allie Goertz wrote a song about the ""The Simpsons"" character Milhouse, who Matt Groening named...",President Richard Nixon,"{'title': ['Lisa Simpson', 'Marge Simpson', 'Bart Simpson', 'Allie Goertz', 'Milhouse Van Houten', 'Los Angeles Read...","{'title': ['Allie Goertz', 'Allie Goertz', 'Allie Goertz', 'Milhouse Van Houten'], 'sent_id': [0, 1, 2, 0]}"
3,5a82171f5542990a1d231f4a,What nationality was James Henry Miller's wife?,American,"{'title': ['Moloch: or, This Gentile World', 'Launceston by-election, 1874', 'Incest: From a Journal of Love', 'Jame...","{'title': ['Peggy Seeger', 'Peggy Seeger', 'Ewan MacColl'], 'sent_id': [0, 1, 0]}"
4,5a84dd955542997b5ce3ff79,"Cadmium Chloride is slightly soluble in this chemical, it is also called what?",alcohol,"{'title': ['Cadmium chloride', 'Water blue', 'Diflucortolone valerate', 'Heptanoic acid', 'Magnesium chloride', 'Eth...","{'title': ['Cadmium chloride', 'Ethanol'], 'sent_id': [1, 0]}"
5,5a7e36045542991319bc9440,"Which tennis player won more Grand Slam titles, Henri Leconte or Jonathan Stark?",Jonathan Stark,"{'title': ['Li Na', 'Williams sisters', 'Henri Leconte', 'Steffi Graf', '1986 Grand Prix German Open', 'Jonathan Sta...","{'title': ['Jonathan Stark (tennis)', 'Jonathan Stark (tennis)', 'Henri Leconte'], 'sent_id': [0, 1, 1]}"
6,5adf44985542993a75d2646d,Which genus of moth in the world's seventh-largest country contains only one species?,Crambidae,"{'title': ['India', 'List of companies of India', 'Eutrapela', 'Geography of India', 'Yoshiyasua', 'Nepita', 'Parect...","{'title': ['Indogrammodes', 'Indogrammodes', 'India', 'India'], 'sent_id': [0, 1, 0, 1]}"
7,5a832c3455429954d2e2ec41,"Who was once considered the best kick boxer in the world, however he has been involved in a number of controversies ...",Badr Hari,"{'title': ['Verano de Escándalo (1998)', 'Triplemanía VII', 'Protection racket', 'E. Gordon Gee', 'Badr Hari', 'Guer...","{'title': ['Global Fighting Championship', 'Global Fighting Championship', 'Badr Hari', 'Badr Hari'], 'sent_id': [1,..."
8,5a7d0db955429909bec76924,"The Dutch-Belgian television series that ""House of Anubis"" was based on first aired in what year?",2006,"{'title': ['House of Anubis', 'Batibot', 'Wolfblood', 'List of House of Anubis episodes', 'Majisuka Gakuen', 'Gradua...","{'title': ['House of Anubis', 'Het Huis Anubis'], 'sent_id': [0, 1]}"
9,5a89372855429951533612e6,What is the length of the track where the 2013 Liqui Moly Bathurst 12 Hour was staged?,6.213 km long,"{'title': ['Mount Panorama Circuit', '2016 Intercontinental GT Challenge', 'Bathurst 12 Hour', '2018 Intercontinenta...","{'title': ['2013 Liqui Moly Bathurst 12 Hour', '2013 Liqui Moly Bathurst 12 Hour', 'Mount Panorama Circuit', 'Mount ..."


In [28]:
# 4) Inspect context and supporting_facts structure in tabular form
rows = []
for i in range(5):
    ex = train_ds[i]
    ctx = ex["context"]
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        sentences = ctx.get("sentences", [])
        ctx_pairs = list(zip(titles, sentences))
    else:
        ctx_pairs = ctx

    for (title, sents) in ctx_pairs:
        rows.append({
            "id": ex["id"],
            "question": ex["question"],
            "doc_title": title,
            "num_sents": len(sents)
        })
ctx_df = pd.DataFrame(rows)
ctx_df.head(20)


,id,question,doc_title,num_sents
0,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,Radio City (Indian radio station),7
1,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,History of Albanian football,4
2,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,Echosmith,9
3,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,Women's colleges in the Southern United States,4
4,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,First Arthur County Courthouse and Jail,1
5,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,Arthur's Magazine,3
6,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,2014–15 Ukrainian Hockey Championship,5
7,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,First for Women,4
8,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,Freeway Complex Fire,4
9,5a7a06935542990198eaf050,Which magazine was started first Arthur's Magazine or First for Women?,William Rast,5


In [ ]:
# 5) Basic statistics: docs per example, sentences per example (normalized context)
import numpy as np

num_docs = []
num_sents = []

for ex in train_ds:
    ctx = ex["context"]
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        sentences = ctx.get("sentences", [])
        ctx_pairs = list(zip(titles, sentences))
    else:
        ctx_pairs = ctx
    num_docs.append(len(ctx_pairs))
    num_sents.append(sum(len(sents) for _, sents in ctx_pairs))

print("Docs per example: min={}, max={}, mean={:.2f}".format(
    min(num_docs), max(num_docs), np.mean(num_docs)))
print("Sents per example: min={}, max={}, mean={:.2f}".format(
    min(num_sents), max(num_sents), np.mean(num_sents)))


Docs per example: min=2, max=2, mean=2.00
Sents per example: min=2, max=2, mean=2.00


In [30]:
# Helper: normalize HF supporting_facts (dict) to list of (title, sent_idx)

def get_gold_supporting_facts(ex: Dict[str, Any]):
    """Return supporting facts as list of (title, sent_idx).

    HF hotpot_qa stores supporting_facts as a dict:
      {'title': [...], 'sent_id': [...]}.
    """
    sf = ex.get("supporting_facts", {})
    if isinstance(sf, dict):
        titles = sf.get("title", [])
        sent_ids = sf.get("sent_id", [])
        return list(zip(titles, sent_ids))
    # Fallback: already a list of pairs
    return list(sf)



In [31]:
# 6) Simple manual "preprocessing": build a naive prompt + answer from one example

from transformers import AutoTokenizer

BASE_MODEL_ID = "meta-llama/Llama-3.2-3B"  # just for tokenizer
HF_TOKEN = os.environ.get("HUGGINGFACE_HUB_TOKEN")

print("Loading base tokenizer for prompt inspection:", BASE_MODEL_ID)
base_tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token

ex = train_ds[0]
q = ex["question"].strip()
ctx = ex["context"]

# Normalize context to a simple multi-line string
if isinstance(ctx, dict):
    titles = ctx.get("title", [])
    sentences = ctx.get("sentences", [])
    parts = []
    for title, sents in zip(titles, sentences):
        parts.append(f"{title}: {' '.join(sents)}")
    context_text = "\n".join(parts)
else:
    parts = []
    for title, sents in ctx:
        parts.append(f"{title}: {' '.join(sents)}")
    context_text = "\n".join(parts)

naive_prompt = f"""You are a knowledgeable assistant. Use the context to answer the question.

Context:
{context_text}

Question: {q}
Answer:"""

print("=== Naive prompt (for base models) ===\n")
print(naive_prompt[:1200], "...", sep="")
print("\n=== Gold short answer ===\n", ex["answer"])


Loading base tokenizer for prompt inspection: meta-llama/Llama-3.2-3B
=== Naive prompt (for base models) ===

You are a knowledgeable assistant. Use the context to answer the question.

Context:
Radio City (Indian radio station): Radio City is India's first private FM radio station and was started on 3 July 2001.  It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).  It plays Hindi, English and regional songs.  It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.  Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.  The Radio station currently plays a mix of Hindi and Regional music.  Abraham Thomas is the CEO of the company.
History of Albanian football: Football in Albania 

## Notes

- Use these cells to sanity-check that the **training prompt format** (system/user layout, context, and `Answer: ...` tail) matches what we use during evaluation.
- If you change `data_preprocessing_hotpotqa.py` or prompt templates, re-run the last cell to see exactly what the model is trained on.


In [32]:
# 7) Deeper check: does preprocess_dataset align with raw HotpotQA fields?

idx = 0
raw_ex = train_ds[idx]
print("RAW QUESTION:\n", raw_ex["question"])
print("\nRAW ANSWER (gold):\n", raw_ex["answer"])

# Show supporting facts and a compact view of context for this example
print("\nSUPPORTING FACTS:")
# Normalize supporting_facts: HF hotpot_qa uses {'title': [...], 'sent_id': [...]}.
sf = raw_ex.get("supporting_facts", {})
if isinstance(sf, dict):
    sf_titles = sf.get("title", [])
    sf_sent_ids = sf.get("sent_id", [])
    sf_pairs = list(zip(sf_titles, sf_sent_ids))
else:
    sf_pairs = list(sf)

for title, sent_idx in sf_pairs:
    print(f"  - {title} [sent {sent_idx}]")

print("\nCONTEXT DOCS (titles only):")
ctx = raw_ex["context"]
if isinstance(ctx, dict):
    titles = ctx.get("title", [])
    sentences = ctx.get("sentences", [])
    ctx_pairs = list(zip(titles, sentences))
else:
    ctx_pairs = ctx

for j, (title, sents) in enumerate(ctx_pairs):
    print(f"  [{j}] {title} ({len(sents)} sents)")

# Reuse proc from previous cell if available; recompute to be safe
proc = preprocess_dataset(raw_ex, llm_tok, max_len=1024, prompt_format=pf, is_train=True)

full_text = llm_tok.decode(proc["input_ids"], skip_special_tokens=False)
label_ids = [tid for tid in proc["labels"] if tid != -100]
answer_text = llm_tok.decode(label_ids, skip_special_tokens=False)

print("\n===== FULL TRAINING TEXT (prompt + answer) =====\n")
print(full_text)
print("\n===== TARGET ANSWER SEGMENT (labels != -100) =====\n")
print(repr(answer_text))


RAW QUESTION:
 Which magazine was started first Arthur's Magazine or First for Women?

RAW ANSWER (gold):
 Arthur's Magazine

SUPPORTING FACTS:
  - Arthur's Magazine [sent 0]
  - First for Women [sent 0]

CONTEXT DOCS (titles only):
  [0] Radio City (Indian radio station) (7 sents)
  [1] History of Albanian football (4 sents)
  [2] Echosmith (9 sents)
  [3] Women's colleges in the Southern United States (4 sents)
  [4] First Arthur County Courthouse and Jail (1 sents)
  [5] Arthur's Magazine (3 sents)
  [6] 2014–15 Ukrainian Hockey Championship (5 sents)
  [7] First for Women (4 sents)
  [8] Freeway Complex Fire (4 sents)
  [9] William Rast (5 sents)

===== FULL TRAINING TEXT (prompt + answer) =====

 Regional music.  Abraham Thomas is the CEO of the company.
History of Albanian football: Football in Albania existed before the Albanian Football Federation (FSHF) was created.  This was evidenced by the team's registration at the Balkan Cup tournament during 1929-1931, which started in 1

## Model prompting exploration (4 base models)

In this section we **do not** use our fine-tuned checkpoints.
Instead, we:

- Load 4 base HF models (Llama 3.2, Llama 3.1, Qwen base, Mistral v0.3).
- Use our HotpotQA prompt builders (`create_prompt_*`) to see their chat style.
- Run 3–5 validation examples per model and print the **prompt + raw output**.

This is just to understand how each base model behaves on HotpotQA-style prompts before (re)finetuning.


In [39]:
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_TOKEN = os.environ.get("HUGGINGFACE_HUB_TOKEN")

# Define the 4 base HF models we care about (no local checkpoints here).
# Llamas are meta-llama IDs; Mistral/Qwen use the *new* base models from finetune_full_new_models.sh.
MODEL_CONFIGS = [
    {
        "name": "llama32_3b",
        "model_id": "meta-llama/Llama-3.2-3B",
    },
    {
        "name": "llama31_8b",
        "model_id": "meta-llama/Llama-3.1-8B",
    },
    {
        "name": "qwen_8b_base",
        "model_id": "Qwen/Qwen3-8B-Base",
    },
    {
        "name": "mistral7b_v03",
        "model_id": "mistralai/Mistral-7B-v0.3",
    },
]

def load_base_model(model_id: str):
    print(f"\n>>> Loading base model: {model_id}")
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, token=HF_TOKEN)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        token=HF_TOKEN,
    )
    model.eval()
    model.config.use_cache = True
    return model, tok

print("Model configs defined:")
for cfg in MODEL_CONFIGS:
    print(" -", cfg["name"], "->", cfg["model_id"])


Model configs defined:
 - llama32_3b -> meta-llama/Llama-3.2-3B
 - llama31_8b -> meta-llama/Llama-3.1-8B
 - qwen_8b_base -> Qwen/Qwen3-8B-Base
 - mistral7b_v03 -> mistralai/Mistral-7B-v0.3


In [40]:
# Helper to build a simple HotpotQA prompt (same template for all models)

def build_hotpot_prompt(example: Dict[str, Any]) -> str:
    q = example["question"].strip()
    ctx = example["context"]

    # Normalize context to simple multi-line string
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        sentences = ctx.get("sentences", [])
        parts = []
        for title, sents in zip(titles, sentences):
            parts.append(f"{title}: {' '.join(sents)}")
        context_text = "\n".join(parts)
    else:
        parts = []
        for title, sents in ctx:
            parts.append(f"{title}: {' '.join(sents)}")
        context_text = "\n".join(parts)

    prompt = f"""You are a knowledgeable assistant. Use the context to answer the question.

Context:
{context_text}

Question: {q}
Answer:"""
    return prompt

print("Helper build_hotpot_prompt() defined.")

Helper build_hotpot_prompt() defined.


In [41]:
# Run 3–5 validation samples for each base model and print prompt + output

from torch import inference_mode

NUM_SAMPLES_PER_MODEL = 3
val_examples = [dataset["validation"][i] for i in range(NUM_SAMPLES_PER_MODEL)]

for cfg in MODEL_CONFIGS:
    name = cfg["name"]
    mid = cfg["model_id"]
    try:
        model, tok = load_base_model(mid)
    except Exception as e:
        print(f"\n[WARN] Failed to load {name} ({mid}): {e}")
        continue

    print("\n" + "="*80)
    print(f"MODEL: {name} ({mid})")
    print("="*80)

    for idx, ex in enumerate(val_examples):
        prompt = build_hotpot_prompt(ex)
        inputs = tok(prompt, return_tensors="pt").to(model.device)

        with inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tok.pad_token_id,
            )

        full_text = tok.decode(out[0], skip_special_tokens=True)
        print(f"\n--- Example {idx} ---")
        print("[PROMPT]\n", prompt[:800], "...", sep="")
        print("\n[OUTPUT]\n", full_text[:800], "...", sep="")

    # free memory between models
    del model, tok
    torch.cuda.empty_cache()

print("\nDone running base-model HotpotQA prompts.")


>>> Loading base model: meta-llama/Llama-3.2-3B


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.14it/s]



MODEL: llama32_3b (meta-llama/Llama-3.2-3B)

--- Example 0 ---
[PROMPT]
You are a knowledgeable assistant. Use the context to answer the question.

Context:
Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.
Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as w...

[OUTPUT]
You are a knowledgeable assistant. Use the context to answer the question.

Context:
Ed Wood (film): Ed Wood is a

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.23s/it]



MODEL: llama31_8b (meta-llama/Llama-3.1-8B)

--- Example 0 ---
[PROMPT]
You are a knowledgeable assistant. Use the context to answer the question.

Context:
Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.
Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as w...

[OUTPUT]
You are a knowledgeable assistant. Use the context to answer the question.

Context:
Ed Wood (film): Ed Wood is a

Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  3.29it/s]



MODEL: qwen_8b_base (Qwen/Qwen3-8B-Base)

--- Example 0 ---
[PROMPT]
You are a knowledgeable assistant. Use the context to answer the question.

Context:
Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.
Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as w...

[OUTPUT]
You are a knowledgeable assistant. Use the context to answer the question.

Context:
Ed Wood (film): Ed Wood is a 19

Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.13it/s]



MODEL: mistral7b_v03 (mistralai/Mistral-7B-v0.3)

--- Example 0 ---
[PROMPT]
You are a knowledgeable assistant. Use the context to answer the question.

Context:
Ed Wood (film): Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.
Scott Derrickson: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as w...

[OUTPUT]
You are a knowledgeable assistant. Use the context to answer the question.

Context:
Ed Wood (film): Ed Wood

### What to look for

When you run the last three cells, focus on:

- Does the **training target** (`answer_text`) really look like `"Answer: {gold}"` with no extra junk?
- Does the **eval prompt** structurally match the training prompt (same system/user framing and `Answer:` cue)?
- Are epoch-6 outputs closer to the gold answers than epoch-0, even if EM/F1 is still low?

If any of these are off, we’ll know whether to fix preprocessing, prompting, or evaluation next.


In [42]:
# Prototype: build a target string with Supporting facts + Answer from one example

def build_sup_ans_target(example: Dict[str, Any]) -> str:
    """Construct a textual target combining supporting facts and answer.
    This does NOT change training yet – just for exploration.
    """
    q = example["question"].strip()
    ans = str(example["answer"]).strip()
    ctx = example["context"]

    # Normalize context to list of (title, sents)
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        sentences = ctx.get("sentences", [])
        ctx_pairs = list(zip(titles, sentences))
    else:
        ctx_pairs = ctx

    # Normalize supporting facts from dict -> list of (title, sent_idx)
    sf = example.get("supporting_facts", {})
    if isinstance(sf, dict):
        sf_titles = sf.get("title", [])
        sf_sent_ids = sf.get("sent_id", [])
        sfs = list(zip(sf_titles, sf_sent_ids))
    else:
        sfs = list(sf)

    # Map title -> sentences for quick lookup
    ctx_map = {title: sents for title, sents in ctx_pairs}

    lines = []
    lines.append(f"Question: {q}")
    lines.append("")
    lines.append("Supporting facts:")

    # Build lines like: - [Title] sentence_index: sentence_text
    for title, sent_idx in sfs:
        sents = ctx_map.get(title, [])
        sent_text = sents[sent_idx] if 0 <= sent_idx < len(sents) else "<missing-sentence>"
        lines.append(f"- [Title] {title} [Sent] {sent_idx}: {sent_text}")

    lines.append("")
    lines.append(f"Answer: {ans}")
    return "\n".join(lines)

idx = 0
example = train_ds[idx]
print("=== BUILT TARGET (SUP+ANS) FOR EXAMPLE 0 ===\n")
print(build_sup_ans_target(example))


=== BUILT TARGET (SUP+ANS) FOR EXAMPLE 0 ===

Question: Which magazine was started first Arthur's Magazine or First for Women?

Supporting facts:
- [Title] Arthur's Magazine [Sent] 0: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.
- [Title] First for Women [Sent] 0: First for Women is a woman's magazine published by Bauer Media Group in the USA.

Answer: Arthur's Magazine


In [43]:
# Prototype: parse Supporting facts back from a model-style text output

import re

def parse_supporting_facts(text: str):
    """Parse lines of the form:
    - [Title] {title} [Sent] {idx}: {sentence}
    Returns list of (title, idx).
    """
    sfs = []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("- [Title]"):
            # Example pattern: - [Title] Foo [Sent] 3: some text
            m = re.match(r"^- \[Title\] (.+?) \[Sent\] (\d+):", line)
            if m:
                title = m.group(1).strip()
                idx = int(m.group(2))
                sfs.append((title, idx))
    return sfs

# Quick round-trip check on the synthetic target
synthetic_target = build_sup_ans_target(example)
parsed_sfs = parse_supporting_facts(synthetic_target)

print("\n=== PARSED SUPPORTING FACTS (FROM SYNTHETIC TARGET) ===")
print("Parsed:", parsed_sfs)
# Normalize gold supporting_facts
sf = example.get("supporting_facts", {})
if isinstance(sf, dict):
    sf_titles = sf.get("title", [])
    sf_sent_ids = sf.get("sent_id", [])
    gold_sfs = list(zip(sf_titles, sf_sent_ids))
else:
    gold_sfs = list(sf)
print("Gold:  ", gold_sfs)



=== PARSED SUPPORTING FACTS (FROM SYNTHETIC TARGET) ===
Parsed: [("Arthur's Magazine", 0), ('First for Women', 0)]
Gold:   [("Arthur's Magazine", 0), ('First for Women', 0)]


In [44]:
# Prototype: simple Sup EM/F1 metric on (title, sent_idx) pairs

from collections import Counter

def sup_em_f1(pred_sfs, gold_sfs):
    """Compute EM and F1 over sets of supporting facts (title, idx)."""
    pred = Counter(pred_sfs)
    gold = Counter(gold_sfs)

    # Exact match
    em = 1.0 if pred == gold else 0.0

    # F1 on multiset overlap
    common = sum((pred & gold).values())
    if common == 0:
        return em, 0.0
    prec = common / max(1, sum(pred.values()))
    rec = common / max(1, sum(gold.values()))
    f1 = 2 * prec * rec / (prec + rec)
    return em, f1

# Test Sup metric on the synthetic perfect prediction
# Use the same normalized gold_sfs from above
em, f1 = sup_em_f1(parsed_sfs, gold_sfs)
print("\n=== SUP METRIC ON PERFECT SYNTHETIC OUTPUT ===")
print(f"Sup EM = {em:.3f}, Sup F1 = {f1:.3f}")



=== SUP METRIC ON PERFECT SYNTHETIC OUTPUT ===
Sup EM = 1.000, Sup F1 = 1.000


In [45]:
# 8) Prototype answer-only target format and EM/F1 (self-contained)

import re
import string

# Normalization similar to eval_hotpotqa (Hotpot official style)

def _normalize_text(s: str) -> str:
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))


def answer_em_f1(pred: str, gold: str):
    p = _normalize_text(pred)
    g = _normalize_text(gold)
    em = 1.0 if p == g else 0.0
    p_tokens = p.split()
    g_tokens = g.split()
    if not p_tokens and not g_tokens:
        return em, 1.0
    if not p_tokens or not g_tokens:
        return em, 0.0
    # overlap
    common = {}
    for t in p_tokens:
        if t in g_tokens:
            common[t] = min(common.get(t, 0) + 1, g_tokens.count(t))
    num_same = sum(common.values())
    if num_same == 0:
        return em, 0.0
    prec = num_same / len(p_tokens)
    rec = num_same / len(g_tokens)
    f1 = 2 * prec * rec / (prec + rec)
    return em, f1


def build_answer_only_target(example: Dict[str, Any]) -> str:
    """Very simple training target: just 'Answer: {answer}'"""
    return f"Answer: {str(example['answer']).strip()}"

# Quick sanity check on one example
ex0 = train_ds[0]
ans_target = build_answer_only_target(ex0)
print("=== Answer-only target for example 0 ===")
print(ans_target)

# If we pretend the model outputs exactly this string, EM/F1 should be perfect
pred = ans_target.replace("Answer:", "").strip()
em, f1 = answer_em_f1(pred, str(ex0["answer"]).strip())
print("\nAnswer EM = {:.3f}, F1 = {:.3f}".format(em, f1))



=== Answer-only target for example 0 ===
Answer: Arthur's Magazine

Answer EM = 1.000, F1 = 1.000


## 9) Answer-only vs Sup+Answer probe with Llama-3.2-3B

In this section we:

- Load the **Llama-3.2-3B** base model from HF (same ID used in our finetuning scripts).
- For 20 validation examples:
  - Run an **Answer-only** prompt and compute EM/F1.
  - Run a **Sup+Answer** prompt and compute:
    - Answer EM/F1 (using `answer_em_f1`).
    - Sup EM/F1 (using the `parse_supporting_facts` + `sup_em_f1` prototype).

This is still exploration only. It tells us which target format makes more sense *before* we change `data_preprocessing_hotpotqa.py` / `finetuning_hotpotqa.py` / `eval_hotpotqa.py`.


In [46]:
# 9a) Load Llama-3.2-3B base model for probing

from transformers import AutoModelForCausalLM, AutoTokenizer

LLAMA32_ID = "meta-llama/Llama-3.2-3B"
HF_TOKEN = os.environ.get("HUGGINGFACE_HUB_TOKEN")

print("Loading Llama-3.2-3B base model for probing:", LLAMA32_ID)
llama_tok = AutoTokenizer.from_pretrained(LLAMA32_ID, trust_remote_code=True, token=HF_TOKEN)
if llama_tok.pad_token is None:
    llama_tok.pad_token = llama_tok.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA32_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    token=HF_TOKEN,
)
llama_model.eval()
llama_model.config.use_cache = True

print("Llama-3.2-3B loaded.")


Loading Llama-3.2-3B base model for probing: meta-llama/Llama-3.2-3B


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.13it/s]

Llama-3.2-3B loaded.


In [49]:
# 9b) Answer-only probing: 20 validation examples

from torch import inference_mode

def build_answer_only_prompt(example: Dict[str, Any]) -> str:
    q = example["question"].strip()
    ctx = example["context"]

    # Normalize context to simple multi-line string
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        sentences = ctx.get("sentences", [])
        parts = []
        for title, sents in zip(titles, sentences):
            parts.append(f"{title}: {' '.join(sents)}")
        context_text = "\n".join(parts)
    else:
        parts = []
        for title, sents in ctx:
            parts.append(f"{title}: {' '.join(sents)}")
        context_text = "\n".join(parts)

    prompt = f"""You are a knowledgeable assistant. Use the context to answer the question.

Context:
{context_text}

Question: {q}

Please answer with a single short phrase in this exact format:
Answer: [SHORT_ANSWER]
"""
    return prompt


def extract_answer_from_text(text: str) -> str:
    # Try to find 'Answer:' and take the text after it on the same line
    if "Answer:" in text:
        tail = text.split("Answer:", 1)[1].strip()
        # Take up to first newline
        return tail.splitlines()[0].strip()
    # Fallback: last non-empty line
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return lines[-1] if lines else ""

N = 20
val_subset = [dataset["validation"][i] for i in range(N)]

ans_em_sum = 0.0
ans_f1_sum = 0.0

print(f"Running Answer-only probe on {N} validation samples...")

for idx, ex in enumerate(val_subset):
    prompt = build_answer_only_prompt(ex)
    inputs = llama_tok(prompt, return_tensors="pt").to(llama_model.device)

    with inference_mode():
        out = llama_model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            temperature=1.0,
            pad_token_id=llama_tok.pad_token_id,
        )

    full_text = llama_tok.decode(out[0], skip_special_tokens=True)
    pred_ans = extract_answer_from_text(full_text)
    gold_ans = str(ex["answer"]).strip()

    em, f1 = answer_em_f1(pred_ans, gold_ans)
    ans_em_sum += em
    ans_f1_sum += f1

    print(f"\n--- Example {idx} ---")
    print("Q:", ex["question"])
    print("Gold:", gold_ans)
    print("Pred:", pred_ans)
    print("EM={:.3f}, F1={:.3f}".format(em, f1))

print("\n=== Answer-only aggregate over {} samples ===".format(N))
print("EM={:.3f}, F1={:.3f}".format(ans_em_sum / N, ans_f1_sum / N))


Running Answer-only probe on 20 validation samples...

--- Example 0 ---
Q: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Pred: [SHORT_ANSWER]
EM=0.000, F1=0.000

--- Example 1 ---
Q: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold: Chief of Protocol
Pred: [SHORT_ANSWER]
EM=0.000, F1=0.000

--- Example 2 ---
Q: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Gold: Animorphs
Pred: [SHORT_ANSWER]
EM=0.000, F1=0.000

--- Example 3 ---
Q: Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?
Gold: no
Pred: [SHORT_ANSWER]
EM=0.000, F1=0.000

--- Example 4 ---
Q: The director of the romantic comedy "Big Stone Gap" is based in what New York city?
Gold: Greenwich Village, New York City
Pred: [SHORT_ANSWER]
EM=0.000, F1=0.000

--- Example 5 ---
Q: 2014 S/S is the debut album of a South

In [50]:
# 9c) Sup+Answer probing: 20 validation examples

# We reuse build_sup_ans_target (for gold format) and parse_supporting_facts + sup_em_f1 defined earlier.

sup_em_sum = 0.0
sup_f1_sum = 0.0
ans2_em_sum = 0.0
ans2_f1_sum = 0.0

print(f"Running Sup+Answer probe on {N} validation samples...")

for idx, ex in enumerate(val_subset):
    # Build context string as in build_hotpot_prompt
    q = ex["question"].strip()
    ctx = ex["context"]
    if isinstance(ctx, dict):
        titles = ctx.get("title", [])
        sentences = ctx.get("sentences", [])
        parts = []
        for title, sents in zip(titles, sentences):
            parts.append(f"{title}: {' '.join(sents)}")
        context_text = "\n".join(parts)
    else:
        parts = []
        for title, sents in ctx:
            parts.append(f"{title}: {' '.join(sents)}")
        context_text = "\n".join(parts)

    # Instruction matching our Sup+Answer target format
    prompt = f"""You are a knowledgeable assistant. Use the context to answer the question.

Context:
{context_text}

Question: {q}

First, list the supporting facts as lines in this exact format:
- [Title] {{TITLE}} [Sent] {{INDEX}}: {{SENTENCE}}
Then, on the last line, output the final short answer as:
Answer: {{SHORT_ANSWER}}
"""

    inputs = llama_tok(prompt, return_tensors="pt").to(llama_model.device)

    with inference_mode():
        out = llama_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=1.0,
            pad_token_id=llama_tok.pad_token_id,
        )

    full_text = llama_tok.decode(out[0], skip_special_tokens=True)

    # Parse supporting facts
    pred_sfs = parse_supporting_facts(full_text)
    # Normalize gold supporting_facts
    sf = ex.get("supporting_facts", {})
    if isinstance(sf, dict):
        sf_titles = sf.get("title", [])
        sf_sent_ids = sf.get("sent_id", [])
        gold_sfs = list(zip(sf_titles, sf_sent_ids))
    else:
        gold_sfs = list(sf)

    sup_em, sup_f1_val = sup_em_f1(pred_sfs, gold_sfs)
    sup_em_sum += sup_em
    sup_f1_sum += sup_f1_val

    # Extract answer same way as answer-only
    pred_ans2 = extract_answer_from_text(full_text)
    gold_ans2 = str(ex["answer"]).strip()
    em2, f12 = answer_em_f1(pred_ans2, gold_ans2)
    ans2_em_sum += em2
    ans2_f1_sum += f12

    print(f"\n--- Example {idx} ---")
    print("Q:", ex["question"])
    print("Gold answer:", gold_ans2)
    print("Pred answer:", pred_ans2)
    print("Sup gold:", gold_sfs)
    print("Sup pred:", pred_sfs)
    print("Ans EM={:.3f}, F1={:.3f} | Sup EM={:.3f}, F1={:.3f}".format(em2, f12, sup_em, sup_f1_val))

print("\n=== Sup+Answer aggregate over {} samples ===".format(N))
print("Answer EM={:.3f}, F1={:.3f}".format(ans2_em_sum / N, ans2_f1_sum / N))
print("Sup EM={:.3f}, F1={:.3f}".format(sup_em_sum / N, sup_f1_sum / N))


Running Sup+Answer probe on 20 validation samples...

--- Example 0 ---
Q: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Pred answer: {SHORT_ANSWER}
Sup gold: [('Scott Derrickson', 0), ('Ed Wood', 0)]
Sup pred: []
Ans EM=0.000, F1=0.000 | Sup EM=0.000, F1=0.000

--- Example 1 ---
Q: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold answer: Chief of Protocol
Pred answer: {SHORT_ANSWER}
Sup gold: [('Kiss and Tell (1945 film)', 0), ('Shirley Temple', 0), ('Shirley Temple', 1)]
Sup pred: []
Ans EM=0.000, F1=0.000 | Sup EM=0.000, F1=0.000

--- Example 2 ---
Q: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Gold answer: Animorphs
Pred answer: {SHORT_ANSWER}
Sup gold: [('The Hork-Bajir Chronicles', 0), ('The Hork-Bajir Chronicles', 1), ('The Hork-Bajir Chronicles', 2), ('Animorphs', 0), ('Animorphs', 1)

### Next step after this exploration

Once you’ve run the 9a–9c cells:

- You’ll have concrete examples of how **Llama-3.2-3B** responds to:
  - A simple **Answer-only** format.
  - A more structured **Sup+Answer** format.
- You’ll have EM/F1 numbers for both formats on 20 samples, which tells us which is easier for the base model to follow.

From there, we can:

1. Pick the better target format (Answer-only vs Sup+Answer) based on clarity + metrics.
2. Port that format into `data_preprocessing_hotpotqa.py`.
3. Align `finetuning_hotpotqa.py` prompts with this format.
4. Align `eval_hotpotqa.py` parsing/metrics (`answer_em_f1`, Sup parser) with the same assumptions.

That will close the loop you described: **finetuning prompt ↔ training target ↔ eval prompt ↔ eval extraction**, all consistent and validated in this notebook first.
